In [1]:
import asyncio, json, uuid, time, structlog
from dataclasses import dataclass, field
from typing import Any, Dict, Optional

structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt='iso'),
        structlog.processors.JSONRenderer()
    ],
    logger_factory=structlog.PrintLoggerFactory(),
)
log = structlog.get_logger()

@dataclass
class Message:
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    body: Dict[str, Any] = field(default_factory=dict)
    delivery_count: int = 0
    dead_lettered_at: Optional[str] = None
    dead_letter_reason: Optional[str] = None

class Broker:
    """Minimal in-process broker simulating RabbitMQ queues."""

    def __init__(self):
        self.queues: Dict[str, asyncio.Queue] = {
            'payments': asyncio.Queue(),
            'payments.dlq': asyncio.Queue(),
        }
        self.stats = {'published': 0, 'consumed': 0, 'dead_lettered': 0}

    async def publish(self, queue_name: str, body: dict):
        msg = Message(body=body)
        await self.queues[queue_name].put(msg)
        self.stats['published'] += 1
        log.info('mq.published', queue=queue_name, message_id=msg.id, body=body)
        return msg.id

    async def consume(self, queue_name: str) -> Optional[Message]:
        try:
            return self.queues[queue_name].get_nowait()
        except asyncio.QueueEmpty:
            return None

    async def dead_letter(self, msg: Message, reason: str):
        msg.dead_lettered_at = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
        msg.dead_letter_reason = reason
        await self.queues['payments.dlq'].put(msg)
        self.stats['dead_lettered'] += 1
        log.warning('mq.dead_lettered', message_id=msg.id, reason=reason)

    def depth(self, queue_name: str) -> int:
        return self.queues[queue_name].qsize()

broker = Broker()
print('✅ In-memory broker ready — queues: payments, payments.dlq')

✅ In-memory broker ready — queues: payments, payments.dlq


In [2]:
"""---
## Part 3 — FastAPI Producer

`POST /payments` validates the payload and publishes a payment event to the queue. The response is immediate — the caller does **not** wait for fraud check or processing.
"""

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager

class PaymentRequest(BaseModel):
    amount: float = Field(..., gt=0, description='Amount in GBP')
    currency: str = Field(default='GBP')
    account_id: str
    reference: Optional[str] = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    log.info('app.startup', message='Connecting to broker...')
    # In production: await aio_pika.connect_robust(RABBIT_URL)
    yield
    log.info('app.shutdown', message='Disconnecting from broker...')

app = FastAPI(title='EY Payment Queue API', version='2.0.0', lifespan=lifespan)

@app.post('/payments', status_code=202)
async def enqueue_payment(payment: PaymentRequest):
    """Accept a payment and publish it to the queue. Returns immediately."""
    msg_id = await broker.publish('payments', payment.model_dump())
    return {
        'message_id': msg_id,
        'status': 'queued',
        'queue_depth': broker.depth('payments')
    }

print('✅ Producer endpoint /payments defined')


✅ Producer endpoint /payments defined


In [3]:
"""---
## Part 4 — Consumer Worker with Retry + DLQ

The worker pulls messages off the queue, processes them, and:
- **ACKs** on success (message gone)
- **NACKs** on transient failure (delivery count < 3) → requeued with backoff
- **Dead-letters** on persistent failure (delivery count ≥ 3)
"""

import random

MAX_DELIVERIES = 3

async def process_payment(msg: Message) -> dict:
    """Simulate payment processing. Fails 40% of the time."""
    await asyncio.sleep(0.05)  # simulate work
    if random.random() < 0.4:
        raise ValueError(f'Processor error — fraud check timeout for {msg.body.get("account_id")}')
    return {'processed_id': str(uuid.uuid4()), 'status': 'settled'}


async def worker_tick():
    """Process one message from the queue. Call repeatedly to drain."""
    msg = await broker.consume('payments')
    if msg is None:
        return None  # queue empty

    msg.delivery_count += 1
    log.info('worker.processing',
             message_id=msg.id,
             delivery_count=msg.delivery_count,
             account=msg.body.get('account_id'))

    try:
        result = await process_payment(msg)
        broker.stats['consumed'] += 1
        log.info('worker.acked', message_id=msg.id, result=result)
        return {'acked': msg.id, 'result': result}

    except Exception as e:
        if msg.delivery_count >= MAX_DELIVERIES:
            await broker.dead_letter(msg, reason=str(e))
            return {'dead_lettered': msg.id, 'reason': str(e)}
        else:
            # NACK: re-queue for retry
            await asyncio.sleep(0.5 * msg.delivery_count)  # backoff
            await broker.queues['payments'].put(msg)
            log.warning('worker.nacked',
                        message_id=msg.id,
                        attempt=msg.delivery_count,
                        error=str(e))
            return {'nacked': msg.id, 'attempt': msg.delivery_count}


@app.get('/payments/worker')
async def drain_one():
    """Manually trigger one worker tick (for demo). In production, runs as a background task."""
    result = await worker_tick()
    if result is None:
        return {'status': 'queue_empty', 'depth': broker.depth('payments')}
    return result


print('✅ Consumer worker + /payments/worker endpoint defined')

✅ Consumer worker + /payments/worker endpoint defined


In [4]:
"""---
## Part 5 — Health Checks

Kubernetes probes need `/health/live` and `/health/ready`. The readiness check verifies the broker is reachable before accepting traffic.
"""

@app.get('/health/live')
async def liveness():
    return {'status': 'alive'}


@app.get('/health/ready')
async def readiness():
    checks = {}

    # Check broker connectivity (in-memory: always ok; swap for real AMQP check)
    try:
        _ = broker.depth('payments')  # Real impl: await connection.channel()
        checks['mq'] = 'ok'
    except Exception as e:
        checks['mq'] = f'error: {e}'

    # Stub DB check
    checks['db'] = 'ok'

    all_ok = all(v == 'ok' for v in checks.values())
    status_code = 200 if all_ok else 503

    return JSONResponse(
        content={'status': 'ready' if all_ok else 'not_ready', **checks,
                 'queue_depth': broker.depth('payments')},
        status_code=status_code
    )

print('✅ Health endpoints defined')

✅ Health endpoints defined


In [5]:
"""---
## Part 6 — Admin DLQ Endpoints
"""

from fastapi import Query

@app.get('/admin/dlq')
async def inspect_dlq(limit: int = Query(default=10, le=50)):
    """Return up to `limit` messages sitting in the DLQ."""
    items = []
    # Peek without consuming
    temp = []
    while len(items) < limit:
        msg = await broker.consume('payments.dlq')
        if msg is None:
            break
        items.append({
            'id': msg.id,
            'body': msg.body,
            'delivery_count': msg.delivery_count,
            'dead_lettered_at': msg.dead_lettered_at,
            'reason': msg.dead_letter_reason
        })
        temp.append(msg)

    # Put messages back (peek semantics)
    for m in temp:
        await broker.queues['payments.dlq'].put(m)

    return {'dlq_depth': broker.depth('payments.dlq'), 'messages': items}


@app.post('/admin/dlq/retry')
async def replay_dlq(limit: int = Query(default=5, le=20)):
    """Replay up to `limit` DLQ messages back to the main payments queue."""
    replayed = []
    for _ in range(limit):
        msg = await broker.consume('payments.dlq')
        if msg is None:
            break
        msg.delivery_count = 0  # reset retries
        msg.dead_lettered_at = None
        msg.dead_letter_reason = None
        await broker.queues['payments'].put(msg)
        replayed.append(msg.id)
        log.info('dlq.replayed', message_id=msg.id)

    return {'replayed': len(replayed), 'message_ids': replayed,
            'payments_depth': broker.depth('payments')}


@app.get('/admin/stats')
async def queue_stats():
    return {
        **broker.stats,
        'payments_depth': broker.depth('payments'),
        'dlq_depth': broker.depth('payments.dlq'),
    }

print('✅ Admin endpoints defined: /admin/dlq, /admin/dlq/retry, /admin/stats')

✅ Admin endpoints defined: /admin/dlq, /admin/dlq/retry, /admin/stats


In [7]:
"""---
## Part 7 — Run & Demo End-to-End Flow
"""

NGROK_TOKEN = '3EtFT46H7xxhBmy1uH5M8UDzre2_48jdnDKvd7sxRNtEGb9kp'  # ← replace this

import threading, uvicorn
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN

thread = threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning'), daemon=True)
thread.start()

import time; time.sleep(2)
tunnel = ngrok.connect(8001)
BASE = tunnel.public_url

print(f'🌐 {BASE}')
print(f'   Docs:     {BASE}/docs')
print(f'   Health:   {BASE}/health/ready')
print(f'   Stats:    {BASE}/admin/stats')

import httpx

with httpx.Client() as c:
    # 1. Check health
    print('--- HEALTH ---')
    print(c.get(f'{BASE}/health/ready').json())

    # 2. Publish 5 payments
    print('\n--- PUBLISH 5 PAYMENTS ---')
    for i in range(5):
        r = c.post(f'{BASE}/payments',
                   json={'amount': (i+1)*100, 'currency': 'GBP', 'account_id': f'ACC-{i+1:03}'})
        print(f'  Payment {i+1}: {r.json()}')

    # 3. Drain the queue — some will DLQ
    print('\n--- DRAIN QUEUE (worker ticks) ---')
    for _ in range(12):  # more ticks than messages to handle retries
        r = c.get(f'{BASE}/payments/worker')
        result = r.json()
        if result.get('status') != 'queue_empty':
            print(f'  {result}')

    # 4. Show stats
    print('\n--- FINAL STATS ---')
    print(c.get(f'{BASE}/admin/stats').json())

    # 5. Inspect DLQ
    print('\n--- DLQ CONTENTS ---')
    print(json.dumps(c.get(f'{BASE}/admin/dlq').json(), indent=2))

    # 6. Replay DLQ
    print('\n--- REPLAY DLQ ---')
    print(c.post(f'{BASE}/admin/dlq/retry').json())

{"message": "Connecting to broker...", "event": "app.startup", "level": "info", "timestamp": "2026-06-09T07:27:14.105971Z"}


ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8001): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted


{"message": "Disconnecting from broker...", "event": "app.shutdown", "level": "info", "timestamp": "2026-06-09T07:27:14.111507Z"}
🌐 https://cucumber-gladly-liability.ngrok-free.dev
   Docs:     https://cucumber-gladly-liability.ngrok-free.dev/docs
   Health:   https://cucumber-gladly-liability.ngrok-free.dev/health/ready
   Stats:    https://cucumber-gladly-liability.ngrok-free.dev/admin/stats
--- HEALTH ---
{'status': 'ready', 'mq': 'ok', 'db': 'ok', 'queue_depth': 0}

--- PUBLISH 5 PAYMENTS ---
{"queue": "payments", "message_id": "426c2045-bed8-4c32-a1a7-8b049f6fbb3a", "body": {"amount": 100.0, "currency": "GBP", "account_id": "ACC-001", "reference": null}, "event": "mq.published", "level": "info", "timestamp": "2026-06-09T07:27:21.433491Z"}
  Payment 1: {'message_id': '426c2045-bed8-4c32-a1a7-8b049f6fbb3a', 'status': 'queued', 'queue_depth': 1}
{"queue": "payments", "message_id": "b7580be1-1bf7-4114-9dba-748cca4eba5e", "body": {"amount": 200.0, "currency": "GBP", "account_id": "ACC-